In [ ]:
# =========================
# nb_aggregate
# =========================

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

from pyspark.sql import functions as F

# ---------- Load canonical ----------
ch = spark.table("canonical_holdings").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)

if ch.rdd.isEmpty():
    print(f"[nb_aggregate] No data for period={period}, run_id={run_id}")
    mssparkutils.notebook.exit("NO_DATA")

# ---------- policy_aggregates ----------
pa = (
    ch.groupBy("period", "run_id", "dfm_id", "policy_id")
      .agg(
          F.sum(F.coalesce(F.col("cash_value_gbp").cast("double"), F.lit(0.0))).alias("total_cash_value_gbp"),
          F.sum(F.coalesce(F.col("bid_value_gbp").cast("double"), F.lit(0.0))).alias("total_bid_value_gbp"),
          F.sum(F.coalesce(F.col("accrued_interest_gbp").cast("double"), F.lit(0.0))).alias("total_accrued_interest_gbp")
      )
      .select(
          "period","run_id","dfm_id","policy_id",
          F.col("total_cash_value_gbp").cast("decimal(28,10)").alias("total_cash_value_gbp"),
          F.col("total_bid_value_gbp").cast("decimal(28,10)").alias("total_bid_value_gbp"),
          F.col("total_accrued_interest_gbp").cast("decimal(28,10)").alias("total_accrued_interest_gbp")
      )
)

pa.write.mode("append").saveAsTable("policy_aggregates")

# ---------- tpir_load_equivalent ----------
# Contract columns from 02_data_contracts.md:
# Policyholder_Number, Security_Code, ISIN, Other_Security_ID, ID_Type, Asset_Name,
# Acq_Cost_in_GBP, Cash_Value_in_GBP, Bid_Value_in_GBP, Accrued_Interest,
# Holding, Loc_Bid_Price, Currency_Local

tpir = (
    ch.select(
        F.col("policy_id").alias("Policyholder_Number"),
        F.col("security_id").alias("Security_Code"),
        F.col("isin").alias("ISIN"),
        F.col("other_security_id").alias("Other_Security_ID"),
        F.col("id_type").alias("ID_Type"),
        F.col("asset_name").alias("Asset_Name"),
        F.lit(None).cast("decimal(28,10)").alias("Acq_Cost_in_GBP"),  # not populated in core ingest
        F.col("cash_value_gbp").cast("decimal(28,10)").alias("Cash_Value_in_GBP"),
        F.col("bid_value_gbp").cast("decimal(28,10)").alias("Bid_Value_in_GBP"),
        F.col("accrued_interest_gbp").cast("decimal(28,10)").alias("Accrued_Interest"),
        F.col("holding").cast("decimal(28,10)").alias("Holding"),
        F.col("local_bid_price").cast("decimal(28,10)").alias("Loc_Bid_Price"),
        F.col("local_currency").alias("Currency_Local")
    )
)

tpir.write.mode("append").saveAsTable("tpir_load_equivalent")

print("[nb_aggregate] Wrote policy_aggregates + tpir_load_equivalent")
mssparkutils.notebook.exit("OK")